In [5]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()


RuntimeError: bitsandbytes is not available in this Kaggle runtime. Enable Internet and install it first, or install it from a local Kaggle wheel/package source.

In [2]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")

# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# YOUR CODE HERE
# --------------
# model.train() 
# --------------
# (Within YOUR CODE HERE)
# Enable training mode so LoRA weights can be updated
model.train()  # puts model in training mode (activates dropout, etc.)【42†L492-L496】

# Load the tokenizer for this model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Read training data and convert to lists
train_df = train.to_pandas()
prompts = train_df["prompt"].tolist()
answers = train_df["answer"].tolist()

# Tokenize and prepare input_ids and labels
input_ids_list = []
labels_list = []
for prompt, ans in zip(prompts, answers):
    # Tokenize prompt and answer (no special tokens by default)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(ans, add_special_tokens=False)["input_ids"]
    # Concatenate prompt and answer into one sequence
    ids = prompt_ids + answer_ids
    # Labels: -100 for prompt tokens, answer_ids for answer tokens
    labels = [-100] * len(prompt_ids) + answer_ids
    input_ids_list.append(ids)
    labels_list.append(labels)

# Pad sequences to max length in batch
max_len = max(len(ids) for ids in input_ids_list)
for i in range(len(input_ids_list)):
    # Pad input_ids and labels with tokenizer.pad_token_id and -100 respectively
    pad_len = max_len - len(input_ids_list[i])
    input_ids_list[i] = input_ids_list[i] + [tokenizer.pad_token_id] * pad_len
    labels_list[i]    = labels_list[i]    + [-100] * pad_len

# Convert lists to PyTorch tensors and create DataLoader
import torch
from torch.utils.data import DataLoader, TensorDataset

input_ids_tensor = torch.tensor(input_ids_list, dtype=torch.long)
labels_tensor    = torch.tensor(labels_list, dtype=torch.long)
dataset = TensorDataset(input_ids_tensor, labels_tensor)
loader = DataLoader(dataset, batch_size=4, shuffle=True)  # adjust batch_size as needed

# Define optimizer (only LoRA parameters require gradients)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# Training loop (single epoch shown; add epochs as needed)
for batch in loader:
    inputs, labels = batch
    inputs = inputs.to(model.device)
    labels = labels.to(model.device)
    # Forward pass with labels; loss is computed only on answer tokens
    outputs = model(input_ids=inputs, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

# Save the trained LoRA adapter (includes adapter_config.json and weights)【53†L211-L216】
model.save_pretrained(OUTPUT_DIR)


# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')